# Wk 3 Homework

Classification

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-14 12:44:14--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.1’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.05s   

2026-10-14 12:44:14 (5.14 MB/s) - ‘course_lead_scoring_2026.csv.1’ saved [278746/278746]



In [3]:
import pandas as pd
df = pd.read_csv("course_lead_scoring_2026.csv")
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [4]:
df.head(10)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
5,referral,manufacturing,employed,africa,43841.0,3,5,0.77,1
6,NaN,education,employed,north_america,58738.0,3,9,0.78,1
7,paid_ads,technology,employed,north_america,95059.0,2,4,0.41,1
8,organic_search,healthcare,employed,asia,62565.0,3,6,0.52,1
9,referral,retail,student,asia,12000.0,2,2,0.35,0


In [5]:
# Fill missing values
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']
num_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

for col in cat_cols:
    df[col] = df[col].fillna('NA')

for col in num_cols:
    df[col] = df[col].fillna(0)

# Question 1

In [6]:
# Most frequent obs in industry col
display(df['industry'].value_counts())

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

# Question 2

In [7]:
df[num_cols].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


# Question 3

In [8]:
feature_cols = df.columns[:-1]
target_col = df.columns[-1]
print(target_col)
print(feature_cols)

converted
Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='str')


In [9]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [10]:
# Mutual information score
from sklearn.metrics import mutual_info_score

for col in cat_cols:
    print(f"With {col}: {mutual_info_score(df_full_train['converted'], df_full_train[col])}")

With lead_source: 0.0347609795100606
With industry: 0.0028984818739526547
With employment_status: 0.02031283511067701
With location: 0.0010498214288891111


# Question 4

In [11]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
train_dict = df_train[cat_cols + num_cols].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)
y_train = df_train[target_col].values

In [12]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [13]:
# Model accuracy on val set
val_dict = df_val[cat_cols+num_cols].to_dict(orient='records')
X_val = dv.transform(val_dict)
y_val = df_val[target_col].values

# Predictions
y_pred = model.predict_proba(X_val)[:, 1]
conversion_decision = (y_pred >= 0.5)
(y_val == conversion_decision).mean()



np.float64(0.645)

# Question 5

In [27]:
feature_set = cat_cols + num_cols
print(feature_set)

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [35]:
# remove one variable at a time and print accuracy on the test set (using full train)
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
print(feature_set)
accuracy_dict = {}
for col in feature_set:
    print(f"Training while removing the feature {col}")
    new_feature_set = list(set(feature_set) - set([col]))
    print(new_feature_set)

    # Train model
    full_train_dict = df_full_train[new_feature_set].to_dict(orient='records')
    X_full_train = dv.fit_transform(full_train_dict)
    y_full_train = df_full_train[target_col].values
    model.fit(X_full_train, y_full_train)

    # predictions
    test_dict = df_test[new_feature_set].to_dict(orient='records')
    X_test = dv.transform(test_dict)
    y_test = df_test[target_col].values
    y_prediction = model.predict_proba(X_test)[:, -1]
    conversion_decision = (y_prediction >= 0.5)
    accuracy_dict[col] = (y_test==conversion_decision).mean()
    print(f"Accuracy of the model with this feature removed: {accuracy_dict[col]}\n")


['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Training while removing the feature lead_source
['annual_income', 'interaction_count', 'industry', 'number_of_courses_viewed', 'employment_status', 'location', 'lead_score']
Accuracy of the model with this feature removed: 0.675

Training while removing the feature industry
['annual_income', 'interaction_count', 'number_of_courses_viewed', 'employment_status', 'lead_source', 'location', 'lead_score']
Accuracy of the model with this feature removed: 0.672

Training while removing the feature employment_status
['annual_income', 'interaction_count', 'industry', 'number_of_courses_viewed', 'lead_source', 'location', 'lead_score']
Accuracy of the model with this feature removed: 0.671

Training while removing the feature location
['annual_income', 'interaction_count', 'industry', 'number_of_courses_viewed', 'employment_status', 'lead_source', 'lead_sco

In [37]:
# With the full set
# Train model
full_train_dict = df_full_train[feature_set].to_dict(orient='records')
X_full_train = dv.fit_transform(full_train_dict)
y_full_train = df_full_train[target_col].values
model.fit(X_full_train, y_full_train)

# predictions
test_dict = df_test[feature_set].to_dict(orient='records')
X_test = dv.transform(test_dict)
y_test = df_test[target_col].values
y_prediction = model.predict_proba(X_test)[:, -1]
conversion_decision = (y_prediction >= 0.5)
model_acc = (y_test==conversion_decision).mean()
print(f"Accuracy of the model: {model_acc}\n")


Accuracy of the model: 0.67



In [39]:
# difference dict
for col in accuracy_dict.keys():
    print(f"Difference with {col}: {model_acc - accuracy_dict[col]}")

Difference with lead_source: -0.0050000000000000044
Difference with industry: -0.0020000000000000018
Difference with employment_status: -0.0010000000000000009
Difference with location: -0.0010000000000000009
Difference with annual_income: -0.051999999999999935
Difference with number_of_courses_viewed: 0.0010000000000000009
Difference with interaction_count: 0.05800000000000005
Difference with lead_score: 0.0


# Question 6

In [40]:
C = [0.000001, 0.00001, 0.0001, 0.001]
model_val_acc = {}

for c in C:
    print(f"Setting C = {c}")
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train) # already defined in cells above

    # validation
    y_pred = model.predict_proba(X_val)[:, 1]
    conversion_decision = (y_pred >= 0.5)
    model_val_acc[c] = (y_val == conversion_decision).mean()
    print(f"Accuracy for c = {c}: {model_val_acc[c]}")


Setting C = 1e-06
Accuracy for c = 1e-06: 0.598
Setting C = 1e-05
Accuracy for c = 1e-05: 0.598
Setting C = 0.0001
Accuracy for c = 0.0001: 0.613
Setting C = 0.001
Accuracy for c = 0.001: 0.645
